# Exploration of LAUS Unemployment Data
 
Data documentation notes available online at: https://download.bls.gov/pub/time.series/la/la.txt
Notes from the documentation:
- "Rates and ratios are expressed as percents with one
decimal place."
- location, survey series both in the series_id
    - la.area file contains the location codes
    - la.series file contains the series codes I *think*


Important codes:
- Unemployment rate measure code: 03
- area_type for counties and equivalents: F
- Period M13 is annual average, the rest are the months in order




In [146]:
# SETUP
import pandas as pd

laus = pd.read_csv(
    "../../data_raw/LAUS/la.data.64.County",
    sep="\t"
)

area = pd.read_csv(
    "../../data_raw/LAUS/la.area",
    sep="\t"
)

series = pd.read_csv(
    "../../data_raw/LAUS/la.series",
    sep="\t"
)

measure = pd.read_csv(
    "../../data_raw/LAUS/la.measure",
    sep="\t"
)

area_type = pd.read_csv(
    "../../data_raw/LAUS/la.area_type",
    sep="\t"
)

period = pd.read_csv(
    '../../data_raw/LAUS/la.period',
    sep="\t"
)

## Main Data Overview + Pre-Clean

### Note:

During my exploratory analysis, I parsed the series_id field directly to better understand the structure of the LAUS identifiers and verify how survey, area, and measure information are encoded. After reviewing the full LAUS documentation in more detail, I realized the la.series and la.area files provide this metadata directly and are meant to be merged with the main data files for this exact purpose. I will be using the offical metadata files in my official cleaning file, but I kept this here to document my exploratory process!

In [147]:
# Checking dataset format
print(laus.head())
print('#################')
print(laus.columns)

# Cleaning up column names
print('#################')
laus.columns = laus.columns.str.strip()


   series_id                       year period         value footnote_codes
0  LAUCN010010000000003            1990    M01           6.5            NaN
1  LAUCN010010000000003            1990    M02           6.4            NaN
2  LAUCN010010000000003            1990    M03           5.6            NaN
3  LAUCN010010000000003            1990    M04           6.6            NaN
4  LAUCN010010000000003            1990    M05           6.1            NaN
#################
Index(['series_id                     ', 'year', 'period', '       value',
       'footnote_codes'],
      dtype='str')
#################


In [148]:
laus_2 = laus.copy()
# Separating series_id into its respective components according to documentation
laus_2['survey'] = laus_2['series_id'].str[:2]
laus_2['adjusted'] = laus_2['series_id'].str[2:3]
laus_2['area_code'] = laus_2['series_id'].str[3:18]
laus_2['measure'] = laus_2['series_id'].str[18:]

print('#################################')
print(laus_2.head())

#################################
                        series_id  year period         value footnote_codes  \
0  LAUCN010010000000003            1990    M01           6.5            NaN   
1  LAUCN010010000000003            1990    M02           6.4            NaN   
2  LAUCN010010000000003            1990    M03           5.6            NaN   
3  LAUCN010010000000003            1990    M04           6.6            NaN   
4  LAUCN010010000000003            1990    M05           6.1            NaN   

  survey adjusted        area_code       measure  
0     LA        U  CN0100100000000  03            
1     LA        U  CN0100100000000  03            
2     LA        U  CN0100100000000  03            
3     LA        U  CN0100100000000  03            
4     LA        U  CN0100100000000  03            


## Preparing and Merging Metadata Files

In [149]:
# AREA FILE PREPARATION

# # File cleanup
# print(area.columns) # all set

# # Restricting to county and equivalent codes
# area = area[area['area_type_code'] == 'F']

# # Separating county and state to use when merging geographic index in
# area[['county', 'state']] = area['area_text'].str.split(', ', expand=True)


# print(area.head())

In [150]:
# SERIES FILE PREPARATION

# Cleaning up file columns
series.columns = series.columns.str.strip()

# Restricting to county/county-equivalent subset
series = series[series['area_type_code'] == 'F']
print(series.head())

                           series_id area_type_code        area_code  \
1186  LAUCN010010000000003                        F  CN0100100000000   
1187  LAUCN010010000000004                        F  CN0100100000000   
1188  LAUCN010010000000005                        F  CN0100100000000   
1189  LAUCN010010000000006                        F  CN0100100000000   
1190  LAUCN010030000000003                        F  CN0100300000000   

      measure_code seasonal  srd_code  \
1186             3        U         1   
1187             4        U         1   
1188             5        U         1   
1189             6        U         1   
1190             3        U         1   

                                   series_title footnote_codes  begin_year  \
1186  Unemployment Rate: Autauga County, AL (U)            NaN        1990   
1187       Unemployment: Autauga County, AL (U)            NaN        1990   
1188         Employment: Autauga County, AL (U)            NaN        1990   
1189    

In [151]:
# Merging metadata files
laus = pd.merge(laus, series, on=('series_id'))

## Exploring and Cleaning Data + Metadata File
Importing file as cleaning in unemployment_cleaning_WIP.py


In [152]:
# Setup
complete = pd.read_csv('../../data_intermediate/unemployment_preclean.csv')

# Inspection
print(complete.describe(include='all'))
print('#############################')
print(complete.info())


KeyboardInterrupt: 

### Notes
- It appears sort_sequence OR srd_code might be state or county fips, so check that when merging in geo id file
- Consider bringing back area_text for graphing

In [ ]:
# Checking States
print(complete['state'].unique())
    # There is one nan state, and while we have PR we do not have DC. Maybe nan is DC?

print('################################')

print(complete[complete['state'].isna()].head(20)) # looks like DC

print('################################')

print(complete[complete['state'].isna()]['county'].unique()) # it is DC! I will add DC as the state abbreviation to match the other files



<StringArray>
['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE',  nan, 'FL', 'GA', 'HI', 'ID',
 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA', 'MI', 'MN', 'MS', 'MO',
 'MT', 'NE', 'NV', 'NH', 'NJ', 'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA',
 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 'PR']
Length: 52, dtype: str
################################
                             series_id  year period         value  \
138427  LAUCN110010000000003            1990    M01           5.5   
138428  LAUCN110010000000003            1990    M02           5.7   
138429  LAUCN110010000000003            1990    M03           5.4   
138430  LAUCN110010000000003            1990    M04           5.3   
138431  LAUCN110010000000003            1990    M05           5.6   
138432  LAUCN110010000000003            1990    M06           6.2   
138433  LAUCN110010000000003            1990    M07           6.3   
138434  LAUCN110010000000003            1990    M08           

In [ ]:
# Pre-aggregation Checks
df = pd.read_csv('../../data_clean/unemployment_WIP.csv')

categoricals = ['year', 'period', 'footnote_codes_x', 'county', 'state']

big_categoricals = ['area_code', 'county', 'srd_code', 'sort_sequence']

# for category in categoricals:
#     print(category + ' unique values:')
#     print(df[category].unique())
#     print('-------------------')

print('#######################')
for category in big_categoricals:
    print(category + ' number of unique values:')
    print(df[category].nunique())
    print('-------------------')

year unique values:
[2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018
 2019 2020 2021 2022 2023 2024 2025 2026]
-------------------
period unique values:


KeyError: 'period'

#### Notes: MYSTERIOUS NUMBER OF COUNTIES

- It looks like srd_code might be state fips (52 unique values with DC and PR)
- It looks like sort_sequence might be county_fips (3225 unique values)
- By a google search, there should be 3,222 counties in the US including DC and PR
    - **what are the three extra counties?**

- County counts:
    - 3,143 in US including DC
    - 78 in PR
    - US, DC, and PR total: 3,221

- Dataset counts:
    - areacodes: 3,184 unique
    - full fips: 3,187 unique




    

## Mysterious Counties Check / FIPS merging

In [ ]:
# Loading in county geo id file and laus file
df = pd.read_csv('../../data_clean/unemployment_WIP.csv')
fips = pd.read_csv('../../data_intermediate/county_geo_id.csv')

# Attempted merge between fips and sort_sequence (test)
# df = pd.merge(df, fips, left_on='sort_sequence', right_on='county_fips', indicator=True)
# NOTE: sort_sequence is NOT county fips! Do not use!

# Merging on county name string instead
# fips = fips.drop(columns = ['_merge'])
# df = pd.merge(df, fips, left_on='county', right_on='COUNTY_NAME', how='outer', indicator=True)



#### Notes
- There are too few counties listed with Puerto Rico (62, should be 78)
- Overall there should be 3,221 counties, but there are only 3,187


In [ ]:
# ######### LOOKING FOR MISSING COUNTIES ##################

# Comparing counties in FIPS file and LAUS file 
geo_fips = set(fips['full_fips'])
laus_fips = set(df['full_fips'])

missing = geo_fips - laus_fips
len(missing) # missing 34 counties

# Finding missing counties
fips.loc[fips['full_fips'].isin(missing), ['full_fips', 'COUNTY_NAME', 'STATE_NAME', 'STATE']].sort_values('STATE')


,full_fips,COUNTY_NAME,STATE_NAME,STATE


### Finding Missing Counties In Original LAUS Area File
(Code below)

#### *Format Issues*
**ISSUE 1: California/Colorado/Honolulu, Hawaii** X
- "x County/city" in LAUS vs "x County" in FIPS
    - san francisco county is in category F but listed "San Francisco County/city"
    - same with Denver and Broomfield in Colorado
    - same with "Honolulu County/city" in LAUS vs "Honolulu County" in FIPS
    - same with "Philadelphia County/city" vs "Philadelphia County" in FIPS

Solution:
- drop "/city" from strings after AK handling

**ISSUE 2: Anchorage, Alaska** X
- "Anchorage Borough/municipality" in LAUS vs "Anchorage Municipality" in FIPS

Solution:
- validation: assert "Borough/municipality" appears only once
- replace "Borough/municipality" string with "Municipality"

**ISSUE 3: Alaska City/Boroughs, Alaska** X
- "Juneau Borough/city" in LAUS vs "Juneau City and Borough" in FIPS
    - Sitka
    - Juneau
    - Wrangell Borough/city
    - Yakutat Borough/city

Solution:
- Handle before "/city" drop
- replace specific string "Borough/city" with "City and Borough"

**ISSUE 4: Massachusets** X
- "Nantucket County/town" in LAUS vs "Nantucket County" in FIPS

Solution:
- Drop "/town" in strings

**ISSUE 5: New Mexico**
- "Dona Ana County" in LAUS vs "Doña Ana County" in FIPS

Solution:
- list of all counties with tilda n in FIPS file
- replace tilda n with non tilda n in this list
- compare with all files as a pre-merge check
- validation: assert there is only one match in the file with the list (unless PR not dropped)
- directly replace non-tilda with tilda version

#### *Content Issues*
**ISSUE 1: Connecticut**
- FIPS file uses counties
- LAUS uses planning regions
- *annual_county_applications and gini have counties only then switch to planning regions only starting in 2022*

Solution:
- Change geo_idx construction file to use planning regions for CT, drop counties
- drop CT counties in annual_county_applications

**ISSUE 2: Kalawao County, Hawaii**
- Kalawao county is missing in LAUS, but it might just be accounted for under Maui County according to google
    - *Google says "Kalawao County: A tiny, isolated quasi-county on the north coast of Molokaʻi. It contains the historic Kalaupapa settlement and has no elected government; it is operated administratively as a judicial district under Maui County."*
- Kalawao is separately accounted for in annual_county_applications
Next steps: 
- Compare with other county-level files to potentially just combine with Maui county across the board
- determine if ommitted in LAUS or if combined with Maui (determines drop vs combine choice, must stay consistent)

Solution:


**ISSSUE 3: Puerto Rico** X
- There are issues with all Municipos with accents in the name merging with LAUS version
- This does not matter bc PR must be dropped; not present in BFS County Applications

Solution:
- Drop PR


In [ ]:
# ################ FINDING MISSING COUNTIES IN ORIGINAL LAUS AREA FILE ###################
# NOTE: make sure code block 57 ('AREA FILE PREPARATION') is commented out in this file before running this block

# Checking number of counties in the F category of original LAUS area file
len(area.loc[area['area_type_code'] == 'F', 'area_text'].unique()) # There are 3,225!!!

# Splitting area_text over comma to separate out state names where applicable
area[['area_1', 'area_2', 'area_3']] = area['area_text'].str.split(', ', expand=True)

### Connecticut is a problem

In [ ]:
############ CHECKING IF PLANNING REGIONS PRESENT IN AREA FILE AT ALL

# Searching for specific text in area file
area[area['area_text'].str.contains('New London', case=False, na=False)]

# Checking LAUS data file pre-FIPS merge for what connecticut looks like
connecticut_laus = df[df['state'] == 'CT']
# NOTE: LAUS somehow does in fact have the planning regions going back to 1990 (full data set)

## After Solving Above Issues, Checking For Remaining FIPS Merge Issues

September 11th - final issue investigation! 

In [ ]:
# Checking merge results for remaining unmatched observations (LAUS-only)

laus_only = df.loc[df['_merge'] == 'left_only']
print(laus_only['county'].unique())

laus_only.groupby('county').agg(
    first_year=('year', 'min'),
    last_year=('year', 'max'),
    n_obs=('year', 'size')
)


<StringArray>
[]
Length: 0, dtype: str


,first_year,last_year,n_obs
county,,,


In [ ]:
# Checking merge results for remaining unmatched observations (FIPS-only) 
fips_only = df.loc[df['_merge'] == 'right_only']
print(fips_only['COUNTY_NAME'].unique())

<StringArray>
[           'Aleutians East Borough',        'Aleutians West Census Area',
            'Anchorage Municipality',                'Bethel Census Area',
               'Bristol Bay Borough',               'Chugach Census Area',
          'Copper River Census Area',                    'Denali Borough',
            'Dillingham Census Area',      'Fairbanks North Star Borough',
                    'Haines Borough',         'Hoonah-Angoon Census Area',
           'Juneau City and Borough',                    'Kalawao County',
           'Kenai Peninsula Borough',         'Ketchikan Gateway Borough',
             'Kodiak Island Borough',              'Kusilvak Census Area',
        'Lake and Peninsula Borough',         'Matanuska-Susitna Borough',
                  'Nome Census Area',               'North Slope Borough',
          'Northwest Arctic Borough',                'Petersburg Borough',
 'Prince of Wales-Hyder Census Area',            'Sitka City and Borough',
           